# 09 — Modelling Dataset and Experimental Protocol

## Purpose

Notebooks 01–08 established the data-quality, temporal, astronomical, and
Panchang-derived feature definitions. This notebook now has a narrower role:

1. construct leakage-safe SOLETE forecasting samples;
2. assemble the nested F0–F4 feature matrices;
3. freeze forecast-valid-time alignment;
4. define expanding-window temporal cross-validation inside the development
   period;
5. lock the terminal June–August 2019 evaluation period;
6. freeze the modelling/evaluation rules **before** revised model fitting.

This notebook deliberately **does not train learned models and does not report
terminal-holdout performance**. Model fitting begins in Notebook 10.

The earlier version of Notebook 09 combined dataset construction,
hyperparameter tuning, terminal-test evaluation, bootstrap analysis, and
result persistence in one notebook. Those historical results are not carried
forward because the astronomy/Panchang conventions and validation design have
since been revised.

## Time notation

For every forecasting sample:

$$
t=\text{forecast issue time},
$$

$$
h=\text{forecast horizon},
$$

and

$$
v=t+h=\text{forecast-valid time}.
$$

Observed predictors may only use measurements available at or before $t$.
Deterministic Gregorian/astronomical/Panchang features may describe $v$.

For SOLETE, the deterministic astronomical state associated with row $v$ is
evaluated at

$$
v_{\mathrm{astro}}=v+2\text{ min}.
$$

## Temporal-validation rationale

Randomly shuffled cross-validation is not appropriate here because later
observations must not be used to train a model that is evaluated on earlier
ones. The revised protocol therefore uses expanding-window / rolling-origin
validation.

Useful references:

- Tashman, L. J. (2000), *Out-of-sample tests of forecasting accuracy: an
  analysis and review*, International Journal of Forecasting 16(4), 437–450,
  DOI `10.1016/S0169-2070(00)00065-0`.
- Hyndman, R. J. & Athanasopoulos, G., *Forecasting: Principles and Practice*,
  3rd ed., Section 5.10, time-series cross-validation / rolling forecasting
  origin.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import swisseph as swe
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete
from src.preprocessing import (
    build_cleaned_solete_features,
    build_gregorian_features,
)
import src.astronomy_features as astronomy_features
import src.vedic_calendar as vedic_calendar
from src.astronomy_features import (
    build_lunar_f3_features,
    build_solar_f2_features,
    load_jpl_ephemeris,
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)
from src.vedic_calendar import (
    build_panchang_f4_raw,
    lahiri_ayanamsha_for_utc_times,
)
from src.feature_families import (
    encode_panchang_fixed_vocabulary,
    PANCHANG_INDICATOR_COUNT,
)

DATA_5MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
EPHEMERIS_FILE = PROJECT_ROOT / "ephemeris" / "de440s.bsp"

for path in [DATA_5MIN, EPHEMERIS_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

# ------------------------------------------------------------------
# Source-parity guard.
#
# Notebook 07 froze pvlib altitude=0 m and delta_t=None.
# Notebook 08 froze standard Lahiri with FLG_NONUT.
#
# Stop immediately if reusable source files still implement the older
# conventions. A source-sync helper is supplied alongside this notebook.
# ------------------------------------------------------------------

if not hasattr(
    astronomy_features,
    "SOLETE_SOLAR_ALTITUDE_CONVENTION_M",
):
    raise RuntimeError(
        "src/astronomy_features.py is older than revised Notebook 07. "
        "Apply the supplied source-sync helper before running Notebook 09."
    )

assert astronomy_features.SOLETE_SOLAR_ALTITUDE_CONVENTION_M == 0.0

assert (
    vedic_calendar.LAHIRI_AYANAMSHA_FLAGS
    & swe.FLG_NONUT
), (
    "src/vedic_calendar.py still uses the older nutation-included "
    "Lahiri convention. Apply the supplied source-sync helper first."
)

df_raw = load_solete(DATA_5MIN).sort_index()

assert df_raw.index.is_monotonic_increasing
assert not df_raw.index.has_duplicates
assert df_raw.index.tz is not None

expected_step = pd.Timedelta(minutes=5)
observed_steps = df_raw.index.to_series().diff().dropna()
assert (observed_steps == expected_step).all(), (
    "SOLETE modelling timeline is not a complete 5-minute grid."
)

print("SOLETE:", df_raw.shape)
print("Period:", df_raw.index.min(), "to", df_raw.index.max())
print("Astronomy offset:", astronomy_features.SOLETE_ASTRONOMY_OFFSET)
print("Solar altitude convention [m]:",
      astronomy_features.SOLETE_SOLAR_ALTITUDE_CONVENTION_M)
print("Lahiri flags:", vedic_calendar.LAHIRI_AYANAMSHA_FLAGS)


## 1. Forecast task and information contract

The primary modelling target remains photovoltaic power:

$$
y^{(h)}(t)=P_{\mathrm{Solar}}(t+h),
\qquad
h\in\{15,30,60\}\text{ min}.
$$

All three horizons are prespecified analysis factors. The revised protocol
does **not** privilege the 60-minute horizon as the sole primary horizon.

The secondary wind target will later reuse the same structure with
`P_Gaia[kW]` as the target/autoregressive power channel. PV and wind remain
separate forecasting tasks.

### F0 daily lag decision

F0 retains the issue-time-aligned previous-day target:

$$
P_{\mathrm{Solar}}(t-24\text{ h}).
$$

This keeps the observational F0 definition identical across forecast
horizons. The separate **daily-persistence baseline** remains target-time
aligned:

$$
\hat P(t+h)=P(t+h-24\text{ h}).
$$

Both are leakage-safe for the selected horizons, but they answer different
questions and are intentionally not conflated.


In [ ]:
RESOLUTION_MINUTES = 5
FORECAST_HORIZONS = {
    15: 3,
    30: 6,
    60: 12,
}

PRIMARY_TARGET = "P_Solar[kW]"

F0_HISTORY_SOURCES = {
    "P_Solar_kw": "P_Solar[kW]",
    "GHI_kw1m2": "GHI[kW1m2]",
    "POA_Irr_kw1m2": "POA Irr[kW1m2]",
    "temperature_degC": "TEMPERATURE[degC]",
    "humidity_clean": "Humidity_clean",
    "pressure_clean_mbar": "Pressure_clean[mbar]",
    "wind_speed_m1s": "WIND_SPEED[m1s]",
    "wind_direction_sin": "wind_direction_sin",
    "wind_direction_cos": "wind_direction_cos",
}

RECENT_LAGS = tuple(range(13))
DAILY_LAG_STEPS = 24 * 60 // RESOLUTION_MINUTES

assert DAILY_LAG_STEPS == 288

df_clean = build_cleaned_solete_features(df_raw)

required_f0_sources = list(F0_HISTORY_SOURCES.values())
assert not df_clean[required_f0_sources].isna().any().any()

# F0: nine channels at t,t-5,...,t-60 plus P_Solar(t-24h).
f0_columns = {}

for feature_name, source_column in F0_HISTORY_SOURCES.items():
    for lag in RECENT_LAGS:
        f0_columns[f"{feature_name}_lag_{lag}"] = (
            df_clean[source_column].shift(lag)
        )

f0_columns["P_Solar_kw_issue_minus_24h"] = (
    df_clean[PRIMARY_TARGET].shift(DAILY_LAG_STEPS)
)

df_f0_full = pd.DataFrame(f0_columns, index=df_clean.index)

assert df_f0_full.shape[1] == 118

first_complete_f0_issue = df_f0_full.dropna().index.min()
expected_first_complete_f0_issue = pd.Timestamp(
    "2018-06-02 00:00:00",
    tz="UTC",
)
assert first_complete_f0_issue == expected_first_complete_f0_issue

print("F0 full-timeline shape:", df_f0_full.shape)
print("First row with complete F0:", first_complete_f0_issue)


## 2. Horizon-specific targets and modelling populations

Each modelling row remains indexed by issue time $t$, but stores its
forecast-valid time $v=t+h$ explicitly.

Rows are retained only when:

- the complete F0 history exists;
- the future target exists;
- the target-valid time lies inside the study interval.

Because F0 includes the issue-time-aligned previous-day target
$P_{\mathrm{Solar}}(t-24\text{ h})$, the first issue time with complete F0
history is **2018-06-02 00:00 UTC**, even though the source dataset begins on
2018-06-01. The first valid modelling timestamp is therefore horizon
dependent.

Split membership is always determined by **valid time**, never issue time.


In [ ]:
STUDY_START = pd.Timestamp("2018-06-01 00:00:00", tz="UTC")
LOCKED_TEST_START = pd.Timestamp("2019-06-01 00:00:00", tz="UTC")
STUDY_END = pd.Timestamp("2019-09-01 00:00:00", tz="UTC")

base_modelling_tables = {}

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():
    table = df_f0_full.copy()

    table["issue_time"] = table.index
    table["valid_time"] = (
        table["issue_time"]
        + pd.Timedelta(minutes=horizon_minutes)
    )
    table["target_P_Solar_kW"] = (
        df_clean[PRIMARY_TARGET].shift(-horizon_steps)
    )
    table["horizon_minutes"] = horizon_minutes

    usable = (
        table[df_f0_full.columns].notna().all(axis=1)
        & table["target_P_Solar_kW"].notna()
        & table["valid_time"].ge(STUDY_START)
        & table["valid_time"].lt(STUDY_END)
    )

    table = table.loc[usable].copy()

    table["final_partition"] = np.where(
        table["valid_time"] < LOCKED_TEST_START,
        "development",
        "locked_test",
    )

    # Direct target-alignment verification.
    sample_positions = np.linspace(
        0, len(table) - 1, 5, dtype=int
    )
    for position in sample_positions:
        valid_time = table["valid_time"].iloc[position]
        observed = table["target_P_Solar_kW"].iloc[position]
        expected = df_clean.loc[valid_time, PRIMARY_TARGET]
        assert np.isclose(observed, expected, atol=0.0, rtol=0.0)

    base_modelling_tables[horizon_minutes] = table

population_summary = []

for horizon, table in base_modelling_tables.items():
    population_summary.append({
        "horizon_minutes": horizon,
        "rows": len(table),
        "development_rows":
            int(table["final_partition"].eq("development").sum()),
        "locked_test_rows":
            int(table["final_partition"].eq("locked_test").sum()),
        "first_issue": table["issue_time"].min(),
        "first_valid": table["valid_time"].min(),
        "last_valid": table["valid_time"].max(),
    })

display(pd.DataFrame(population_summary))


## 3. Deterministic full-timeline feature tables

The deterministic blocks are safely precomputed over the full timestamp
grid because they contain no learned statistics and no future measured
weather.

The row-index convention is:

- F1 Gregorian values describe row timestamp $r$;
- F2/F3/F4 astronomy for row $r$ describes $r+2$ minutes;
- a model sample with valid row $v$ selects deterministic row $v$.

Therefore its astronomical state is automatically evaluated at $v+2$ min.

The source-parity check below also verifies that the reusable F4 module
matches the canonical Notebook 08 Lahiri-NONUT reference.


In [ ]:
ephemeris = load_jpl_ephemeris(EPHEMERIS_FILE)

# F1: no astronomical offset.
df_f1_increment_full = build_gregorian_features(df_clean.index)
assert df_f1_increment_full.shape == (len(df_clean), 15)

# F2/F3: module applies row timestamp + 2 min.
df_f2_increment_full = build_solar_f2_features(df_clean.index)
df_f3_increment_full = build_lunar_f3_features(
    df_clean.index,
    ephemeris,
)

assert df_f2_increment_full.shape == (len(df_clean), 5)
assert df_f3_increment_full.shape == (len(df_clean), 9)

# F4 parents at the same +2 min astronomical reference.
df_tropical_full = tropical_longitudes_for_solete(
    df_clean.index,
    ephemeris,
)

astro_times = solete_astronomy_reference_times(df_clean.index)

ayanamsha = lahiri_ayanamsha_for_utc_times(astro_times)

df_ayanamsha = pd.Series(
    ayanamsha,
    index=df_clean.index,
    name="lahiri_ayanamsha_nonut_deg",
)

# Canonical Notebook 08 first-point guard.
KNOWN_FIRST_LAHIRI_NONUT = 24.11431760121269

assert np.isclose(
    df_ayanamsha.iloc[0],
    KNOWN_FIRST_LAHIRI_NONUT,
    atol=1e-10,
    rtol=0.0,
)

df_f4_raw_full = build_panchang_f4_raw(
    sun_tropical_longitude_deg=
        df_tropical_full["sun_tropical_longitude_deg"],
    moon_tropical_longitude_deg=
        df_tropical_full["moon_tropical_longitude_deg"],
    ayanamsha_deg=df_ayanamsha,
)

assert df_f4_raw_full.shape == (len(df_clean), 5)
assert not df_f4_raw_full.isna().any().any()

first_f4 = df_f4_raw_full.iloc[0]
assert int(first_f4["tithi"]) == 18
assert first_f4["paksha"] == "Krishna"
assert first_f4["karana"] == "Vanija"
assert first_f4["nakshatra"] == "Mula"
assert first_f4["yoga"] == "Shubha"

print("F1 increment:", df_f1_increment_full.shape)
print("F2 increment:", df_f2_increment_full.shape)
print("F3 increment:", df_f3_increment_full.shape)
print("Raw F4:", df_f4_raw_full.shape)
print("First Lahiri NONUT:", df_ayanamsha.iloc[0])
print("Notebook 07/08 source parity: PASSED")


## 4. Assemble valid-time F0–F4 matrices

For a sample issued at $t$:

- F0 is selected from the issue-time row $t$;
- F1 is selected from row $v=t+h$;
- F2/F3/F4 are selected from row $v$, whose astronomy internally represents
  $v+2$ minutes.

F4 uses the fixed 97-column one-hot vocabulary. Category domains are
mathematical definitions, not learned from the modelling population.

### Preserved robustness control: $F3^\ast$

Notebook 08 also defined an optional continuous-parent control:

$$
F3^\ast
=
F3
+
C(\lambda_{\mathrm{Moon,sid}})
+
C(\psi_{\mathrm{Yoga}}).
$$

This adds four continuous variables: sine/cosine of Lahiri sidereal lunar
longitude and sine/cosine of the continuous Yoga parent angle. The Moon–Sun
phase is **not** added again because its sine/cosine representation is already
present in F3.

$F3^\ast$ is **not part of the main F0–F4 hierarchy** and is not included in
the main feature sweep below. It is preserved for a later robustness test that
separates the effect of introducing sidereal continuous parent variables from
the effect of discretizing them into Panchang-derived categories.


In [ ]:
EXPECTED_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}

assert PANCHANG_INDICATOR_COUNT == 97

feature_family_tables = {}
panchang_onehot_tables = {}

for horizon, base in base_modelling_tables.items():
    issue_index = base.index
    valid_index = pd.DatetimeIndex(
        base["valid_time"],
        name="valid_time",
    )

    f0 = df_f0_full.loc[issue_index].copy()

    f1_inc = df_f1_increment_full.loc[valid_index].copy()
    f1_inc.index = issue_index
    f1_inc = f1_inc.add_prefix("valid_")

    f2_inc = df_f2_increment_full.loc[valid_index].copy()
    f2_inc.index = issue_index
    f2_inc = f2_inc.add_prefix("valid_")

    f3_inc = df_f3_increment_full.loc[valid_index].copy()
    f3_inc.index = issue_index
    f3_inc = f3_inc.add_prefix("valid_")

    f4_raw = df_f4_raw_full.loc[valid_index].copy()
    f4_raw.index = issue_index
    f4_raw = f4_raw.add_prefix("valid_")

    f4_onehot = encode_panchang_fixed_vocabulary(f4_raw)

    assert f4_onehot.shape[1] == 97
    assert (f4_onehot.sum(axis=1) == 5).all()

    F0 = f0
    F1 = pd.concat([F0, f1_inc], axis=1)
    F2 = pd.concat([F1, f2_inc], axis=1)
    F3 = pd.concat([F2, f3_inc], axis=1)
    F4 = pd.concat([F3, f4_onehot], axis=1)

    families = {
        "F0": F0,
        "F1": F1,
        "F2": F2,
        "F3": F3,
        "F4": F4,
    }

    for family, table in families.items():
        assert table.index.equals(issue_index)
        assert table.shape[1] == EXPECTED_FEATURE_COUNTS[family]
        assert not table.isna().any().any()

    assert list(F1.columns[:118]) == list(F0.columns)
    assert list(F2.columns[:133]) == list(F1.columns)
    assert list(F3.columns[:138]) == list(F2.columns)
    assert list(F4.columns[:147]) == list(F3.columns)

    feature_family_tables[horizon] = families
    panchang_onehot_tables[horizon] = f4_onehot

print("Nested feature construction: PASSED")

for horizon, families in feature_family_tables.items():
    print(
        horizon,
        {name: table.shape for name, table in families.items()},
    )


## 5. Explicit sample-time audit

A correct feature matrix can still be scientifically wrong if timestamps are
misinterpreted. The audit below makes one sample from each horizon human
readable.

For each selected sample it checks:

$$
v-t=h,
$$

$$
v_{\mathrm{astro}}=v+2\text{ min},
$$

and verifies that the issue-time previous-day F0 feature is exactly

$$
P_{\mathrm{Solar}}(t-24\text{ h}).
$$


In [ ]:
alignment_rows = []

for horizon, base in base_modelling_tables.items():
    row = base.iloc[len(base) // 2]
    issue_time = row["issue_time"]
    valid_time = row["valid_time"]

    issue_minus_24h = issue_time - pd.Timedelta(hours=24)

    stored_daily_lag = (
        feature_family_tables[horizon]["F0"]
        .loc[issue_time, "P_Solar_kw_issue_minus_24h"]
    )
    expected_daily_lag = df_clean.loc[
        issue_minus_24h,
        PRIMARY_TARGET,
    ]

    assert valid_time - issue_time == pd.Timedelta(minutes=horizon)
    assert np.isclose(
        stored_daily_lag,
        expected_daily_lag,
        atol=0.0,
        rtol=0.0,
    )

    alignment_rows.append({
        "horizon_min": horizon,
        "issue_time": issue_time,
        "valid_time": valid_time,
        "astronomy_time":
            valid_time + pd.Timedelta(minutes=2),
        "F0_daily_reference": issue_minus_24h,
    })

display(pd.DataFrame(alignment_rows))


## 6. Rolling-origin folds, target availability, and the locked holdout

The three chronological, expanding-window fold **date ranges** remain unchanged:

| Fold | Training valid-time window | Validation valid-time window |
|---|---|---|
| 1 | [2018-06-01, 2018-12-01) | [2018-12-01, 2019-02-01) |
| 2 | [2018-06-01, 2019-02-01) | [2019-02-01, 2019-04-01) |
| 3 | [2018-06-01, 2019-04-01) | [2019-04-01, 2019-06-01) |

The ranges refer to **forecast-valid times** $v=t+h$. A disjoint valid-time
split alone is not sufficient to guarantee a model could have been fitted by
the first validation issue time. For example, the first 60-minute prediction
valid at 2018-12-01 00:00 is **issued** at 2018-11-30 23:00. Any training
label observed after that issue time would be unavailable at prediction time.

### Fixed-model availability policy

Models are fitted once at the start of each validation fold. We prespecify a
common **60-minute safety gap** $g=\max(h)=60\text{ min}$ across all horizons.
The strict label-availability criterion at the first validation issue is

$$
\max(v_{\mathrm{train}}) <
\min(t_{\mathrm{validation}}),
\qquad t_{\mathrm{validation}}=v_{\mathrm{validation}}-h.
$$

For each fold whose nominal validation start is $B$, training labels must
satisfy $v < B-g$. On a 5-minute grid this withholds exactly **12
training-label uses** per fold. The historical measurements and lag windows
remain physically present; no source observation or feature table is trimmed.

### Validation-scoring and terminal-selection timing

The Fold-1 and Fold-2 validation windows remain complete. Their labels are
available when the **final** hyperparameter selection is made. Only Fold 3
withholds the final 12 valid-time rows from the selection metric:
$v < \text{2019-06-01 00:00}-g$. This guarantees every validation label used
for final selection is available before the earliest terminal-test issue.

After selecting hyperparameters, the final model is refitted using **all
eligible development labels** with
$v < \text{2019-06-01 00:00}-g$; it is frozen before the first
terminal-test forecast is issued.

The terminal test itself remains the full
$[\text{2019-06-01},\text{2019-09-01})$ valid-time interval: **26,496
samples for each horizon**. The terminal period was inspected in historical
experiments; it is locked against *revised* model selection, not described as
previously unseen.

This design conservatively removes **36 fold-specific training-label uses**
(12 per fold) plus **12 Fold-3 validation-scoring uses** per horizon: 48
distinct affected development timestamps, zero deleted observations, and zero
terminal-test exclusions. Because training windows expand, fold counts are not
disjoint observation counts.

This is an **information-availability** check under a fixed-model evaluation
assumption, not a claim that model fitting and hyperparameter selection can be
completed in the five minutes between the last available label and first
forecast issue. A real-time deployment would require separately budgeted
training/selection latency and possibly an earlier model-freeze time.


In [ ]:
from src.forecasting_protocol import (
    CV_FOLDS,
    EXPECTED_CV_COUNTS_BY_HORIZON,
    EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON,
    EXPECTED_UNPURGED_CV_COUNTS_BY_HORIZON,
    MODEL_AVAILABILITY_GAP,
    make_rolling_origin_masks,
    make_terminal_refit_mask,
    terminal_holdout_mask,
    validate_rolling_origin_counts,
)

assert MODEL_AVAILABILITY_GAP == pd.Timedelta(minutes=60)

cv_masks = {}
terminal_refit_masks = {}
cv_summary_rows = []

# Masks operate on the complete, unchanged horizon-specific population.
expected_population = {15: 131325, 30: 131322, 60: 131316}
expected_development = {15: 104829, 30: 104826, 60: 104820}

for horizon, base in base_modelling_tables.items():
    valid_time = base["valid_time"]
    issue_time = base["issue_time"]
    assert len(base) == expected_population[horizon]
    assert int(base["final_partition"].eq("development").sum()) == (
        expected_development[horizon]
    )

    holdout = terminal_holdout_mask(valid_time)
    assert int(holdout.sum()) == 26496
    assert np.array_equal(
        holdout.to_numpy(),
        base["final_partition"].eq("locked_test").to_numpy(),
    )

    final_refit = make_terminal_refit_mask(valid_time)
    assert int(final_refit.sum()) == (
        EXPECTED_TERMINAL_REFIT_COUNTS_BY_HORIZON[horizon]
    )
    assert valid_time.loc[final_refit].max() < issue_time.loc[holdout].min()
    terminal_refit_masks[horizon] = final_refit

    horizon_masks = {}
    fold_masks = make_rolling_origin_masks(valid_time)
    for spec, masks in zip(CV_FOLDS, fold_masks, strict=True):
        fold = spec["fold"]
        train = masks["train_mask"]
        validation = masks["validation_mask"]

        # The learned model must exist before its earliest validation issue.
        assert valid_time.loc[train].max() < issue_time.loc[validation].min()
        assert not (train & validation).any()
        assert not (train & holdout).any()
        assert not (validation & holdout).any()

        historical_train = (
            valid_time.ge(spec["train_start"])
            & valid_time.lt(spec["train_end"])
        )
        historical_validation = (
            valid_time.ge(spec["validation_start"])
            & valid_time.lt(spec["validation_end"])
        )
        train_removed = int((historical_train & ~train).sum())
        validation_removed = int((historical_validation & ~validation).sum())
        assert train_removed == 12
        assert validation_removed == (12 if fold == 3 else 0)
        assert (
            int(historical_train.sum()),
            int(historical_validation.sum()),
        ) == EXPECTED_UNPURGED_CV_COUNTS_BY_HORIZON[horizon][fold - 1]

        horizon_masks[fold] = {"train": train, "validation": validation}
        cv_summary_rows.append({
            "horizon_minutes": horizon,
            "fold": fold,
            "train_rows": int(train.sum()),
            "validation_rows": int(validation.sum()),
            "purged_train_labels": train_removed,
            "withheld_validation_scores": validation_removed,
            "last_training_valid": valid_time.loc[train].max(),
            "first_validation_issue": issue_time.loc[validation].min(),
            "last_scored_validation_valid": valid_time.loc[validation].max(),
        })

    assert valid_time.loc[horizon_masks[3]["validation"]].max() < (
        issue_time.loc[holdout].min()
    )
    assert validate_rolling_origin_counts(valid_time, horizon) == (
        EXPECTED_CV_COUNTS_BY_HORIZON[horizon]
    )
    cv_masks[horizon] = horizon_masks

cv_summary = pd.DataFrame(cv_summary_rows)
display(cv_summary)
print("Availability-safe rolling-origin masks: PASSED")
print("Original observational and modelling grids: PRESERVED")
print("Terminal holdout rows per horizon: 26496")


## 7. Leakage and information-availability audit

The following rules are frozen **before** revised fitting:

- measured predictors are taken from issue time $t$ or earlier;
- deterministic F1–F4 features describe valid time $v=t+h$, with SOLETE
  astronomy evaluated at $v+2$ minutes;
- complete observational grids and all F0–F4 feature/target tables are retained;
- train/validation fold membership and 60-minute purge use **valid time**;
- the **latest training target-valid time is strictly earlier than the first
  validation forecast issue time** in every fold;
- only Fold-3's last 12 validation labels are withheld from the **selection
  metric**, so all selected-validation outcomes precede the earliest test issue;
- final refitting uses only development labels strictly earlier than the
  earliest terminal forecast issue;
- fixed Panchang vocabularies may be computed globally because their category
  definitions contain no learned statistics;
- learned transformations (including scalers) must be fitted **only** on the
  eligible training rows of each fold, and refitted on the eligible development
  population after model selection;
- the terminal holdout and old frozen-model results cannot influence revised
  hyperparameters, feature selection, or preprocessing choices;
- QC flags are not model predictors.

Excluding training labels **does not remove historical observations** from the
lagged input sequence. A validation forecast may legitimately use measurements
at or before its own issue time, including timestamps not selected as labels.


In [ ]:
# Confirm forecast semantics and availability for each horizon.
for horizon, base in base_modelling_tables.items():
    issue = base["issue_time"]
    valid = base["valid_time"]
    assert (valid > issue).all()
    assert (valid - issue == pd.Timedelta(minutes=horizon)).all()

    # Original historical lags remain historical and present in F0.
    assert (issue - pd.Timedelta(minutes=60) <= issue).all()
    assert (issue - pd.Timedelta(hours=24) <= issue).all()
    assert (valid - pd.Timedelta(hours=24) <= issue).all()

    expected_holdout = valid.ge(LOCKED_TEST_START)
    assert np.array_equal(
        expected_holdout.to_numpy(),
        base["final_partition"].eq("locked_test").to_numpy(),
    )
    earliest_test_issue = issue.loc[expected_holdout].min()

    # No training target can come from the future of a fold's first issue.
    for fold in (1, 2, 3):
        train_mask = cv_masks[horizon][fold]["train"]
        validation_mask = cv_masks[horizon][fold]["validation"]
        assert valid.loc[train_mask].max() < issue.loc[validation_mask].min()

    # CV selection and final refit are finished before the first test issue.
    assert valid.loc[cv_masks[horizon][3]["validation"]].max() < (
        earliest_test_issue
    )
    assert valid.loc[terminal_refit_masks[horizon]].max() < (
        earliest_test_issue
    )

    # Preserve the underlying complete 5-minute sample population.
    assert base.index.equals(feature_family_tables[horizon]["F0"].index)
    assert len(base) == len(feature_family_tables[horizon]["F4"])
    assert (base.index.to_series().diff().dropna() == (
        pd.Timedelta(minutes=5)
    )).all()

print("Forecast-time and label-availability audit: PASSED")


## 8. Baselines, models, scaling, and tuning rules for the next notebooks

### Non-learned baselines

For each horizon we retain:

**Persistence**

$$
\hat P(t+h)=P(t)
$$

and **daily persistence**

$$
\hat P(t+h)=P(t+h-24\text{ h}).
$$

A training-fold mean may be retained only as a pipeline sanity check.

### Core learned model suite

The revised core suite is intentionally compact:

1. **Ridge Regression** — linear regularized model;
2. **Random Forest** — bagged nonlinear tree ensemble;
3. **LightGBM** — gradient-boosted tree model;
4. **LSTM** — recurrent neural sequence model;
5. **TCN** — temporal convolutional neural model.

Persistence baselines do not count as trained models. Histogram Gradient
Boosting from the historical Notebook 09 is not part of the revised core
suite; it may remain a secondary sensitivity model if useful later.

### Hyperparameter selection

For every model × horizon × feature family, candidate hyperparameters are
evaluated on the same three temporal folds.

The selection score is the **unweighted mean of fold validation MAE**:

$$
\overline{\mathrm{MAE}}_{\mathrm{CV}}
=
\frac{1}{3}
\sum_{k=1}^{3}
\mathrm{MAE}_k.
$$

Fold-specific MAE and its spread must also be retained. Search spaces must be
fixed before comparing feature-family performance.

### Availability-aware final refit

After choosing hyperparameters using only the three availability-safe folds,
refit once on the complete eligible development population with
$v < \text{2019-06-01 00:00 UTC} - 60\text{ min}$, preserving the identical
feature-family definitions. Freeze that final model **before** issuing its
first terminal forecast. No updating from holdout outcomes is permitted.

A shared 60-minute cutoff is intentionally conservative for 15/30-minute
horizons. Their usable training/validation masks are still evaluated using
the same valid-time semantics and all three horizons remain prespecified.

### Scaling

Any fitted scaler is trained **inside each fold's training period only**.

For Ridge, continuous predictors will be standardized while the fixed
Panchang one-hot indicators remain binary 0/1. Tree models do not require
standardization. Neural-network preprocessing will be fitted using the
corresponding training fold only.

The same semantic feature family must be presented to every architecture.


## 9. Evaluation quantities and the main estimand

The primary error metric is MAE in kW:

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_i
|y_i-\hat y_i|.
$$

RMSE is reported as a complementary measure. For PV, capacity-normalized
MAE/RMSE may be reported relative to the nominal 10 kW system.

All-time performance is the primary operational scope. Daylight-only
performance, defined deterministically by valid-time solar elevation
$>0^\circ$, is a secondary scope.

The central representation effect is written as

$$
\Delta_{\mathrm{MAE}}^{F3\rightarrow F4}
=
\mathrm{MAE}_{F3}
-
\mathrm{MAE}_{F4}.
$$

Thus:

- $\Delta_{\mathrm{MAE}}>0$: F4 has lower error;
- $\Delta_{\mathrm{MAE}}<0$: F3 has lower error;
- values near zero indicate little practical separation.

Relative percentage change may also be reported, but the signed kW
difference remains directly interpretable.

No improvement is assumed in advance.


## 10. Wind extension policy

The present notebook constructs the primary PV modelling population.

The later wind task will reuse the same protocol with three target-specific
changes:

- target and autoregressive power channel become `P_Gaia[kW]`;
- previous-day power lag becomes `P_Gaia(t-24 h)`;
- a wind sample is excluded if the future Gaia target timestamp or any Gaia
  autoregressive-history timestamp required by F0 is quality-flagged.

The common meteorological history, F1–F4 deterministic representations,
horizons, and temporal CV philosophy remain unchanged.

`P_Solar[kW]` should not be silently introduced into the wind F0, and
`P_Gaia[kW]` should not be silently introduced into the PV F0.


In [ ]:
protocol_manifest = {
    "target": PRIMARY_TARGET,
    "resolution_minutes": RESOLUTION_MINUTES,
    "horizons_minutes": tuple(FORECAST_HORIZONS),
    "recent_history_minutes": 60,
    "recent_history_positions": len(RECENT_LAGS),
    "F0_daily_lag": "P_Solar(t-24h), issue-time aligned",
    "feature_counts": EXPECTED_FEATURE_COUNTS,
    "astronomy_row_offset_minutes": 2,
    "panchang_encoding":
        "fixed full one-hot, 97 indicators",
    "development_end_exclusive":
        str(LOCKED_TEST_START),
    "locked_test":
        "[2019-06-01, 2019-09-01) by valid time",
    "temporal_cv":
        "3-fold expanding-window / rolling-origin",
    "fixed_model_availability_gap_minutes": 60,
    "fold_training_purged_labels_each": 12,
    "fold_3_validation_scoring_withheld_labels": 12,
    "fold_1_2_validation_scoring_withheld_labels": 0,
    "terminal_refit_label_cutoff_exclusive":
        str(LOCKED_TEST_START - MODEL_AVAILABILITY_GAP),
    "terminal_test_rows_each_horizon": 26496,
    "cv_selection_metric":
        "unweighted mean validation MAE",
    "core_models": (
        "Ridge",
        "RandomForest",
        "LightGBM",
        "LSTM",
        "TCN",
    ),
}

display(
    pd.Series(protocol_manifest, name="value").to_frame()
)


## Final checkpoint

Notebook 09 now freezes the **dataset and protocol**, not model results.

Before Notebook 10 begins:

- F0–F4 dimensions and nesting are verified;
- issue/valid/astronomical timing is explicit;
- the revised Notebook 07/08 source conventions are guarded;
- all three horizons are treated as prespecified factors;
- temporal model selection uses three availability-safe expanding folds;
- a 60-minute label gap protects each fold's first forecast issue;
- only Fold-3's last 12 validation labels are withheld from selection;
- the final refit uses development labels available before test issue;
- no original timestamps, features, or terminal-test samples are removed;
- scalers/learned preprocessing must be fit inside each fold;
- June–August 2019 is locked against revised tuning;
- no historical Notebook 09 performance result is reused as a revised result;
- the optional $F3^\ast$ continuous-parent robustness control is preserved for
  later representation analysis.

Notebook 10 can therefore begin classical-model selection using only the
development folds. Terminal-holdout evaluation should occur only after the
revised model/hyperparameter choices are frozen.
